# Ollama API Testing Notebook

This notebook tests your Ollama inference service deployed on GKE with KEDA autoscaling.

## Prerequisites
- GKE cluster with Ollama deployment running (see README.md for setup)
- KEDA HTTP interceptor configured
- Models pre-loaded via `./scripts/seed-initial-model.sh`

## Important Notes
- **First request may take 2-4 minutes** if pod is scaled to zero (KEDA will trigger scale-up)
- Modify `OLLAMA_ENDPOINT` below to match your deployment endpoint
- Use environment variables for security (see setup cell)

In [ ]:
import requests
import json
import time
import os
from urllib.parse import urlparse

# =============================================================================
# OLLAMA API CLIENT CONFIGURATION
# =============================================================================
# This section sets up the connection parameters for communicating with Ollama.
# The endpoint can be configured three ways (in order of precedence):
# 1. Environment variable: OLLAMA_ENDPOINT (recommended for production)
# 2. Direct assignment (useful for local testing)
# 3. Default: http://localhost:11434 (for local development)

OLLAMA_ENDPOINT = os.getenv("OLLAMA_ENDPOINT", "http://localhost:11434")
# Alternative configurations (uncomment to use):
# - Local port-forward (Ollama service): 
#   In a separate terminal, run:
#     kubectl port-forward svc/ollama-service 11434:80
#   Then set: OLLAMA_ENDPOINT=http://localhost:11434
#
# - KEDA HTTP Interceptor (requires KEDA HTTP Add-on):
#   In a separate terminal, run:
#     kubectl port-forward -n keda svc/keda-http-add-on-interceptor-proxy 9090:8081 &
#   Then, in this notebook, set:
#     OLLAMA_ENDPOINT="http://localhost:9090"
#
# - Remote GKE: Get external IP and set OLLAMA_ENDPOINT environment variable

MODEL_NAME = "gemma2:2b"  # Lightweight model suitable for demonstrations
TIMEOUT = 60  # Request timeout in seconds (increase for slower networks)

# =============================================================================
# ENDPOINT VALIDATION
# =============================================================================
# Validate the endpoint URL format before making requests to avoid cryptic
# connection errors. This catches common mistakes early (missing protocol, host, etc.)

try:
    parsed = urlparse(OLLAMA_ENDPOINT)
    # Check that endpoint has both scheme (http://) and netloc (hostname:port)
    if not parsed.scheme or not parsed.netloc:
        raise ValueError(f"Invalid endpoint format: {OLLAMA_ENDPOINT}")
    print(f"[OK] Endpoint validated: {OLLAMA_ENDPOINT}")
except Exception as e:
    # Print warning but continue - endpoint might still work despite validation concerns
    print(f"[WARNING] Endpoint validation warning: {e}")

print(f"[INFO] Configuration:")
print(f"   Endpoint: {OLLAMA_ENDPOINT}")
print(f"   Model: {MODEL_NAME}")
print(f"   Timeout: {TIMEOUT}s")

## Step 1: Setup & Configuration

Configure your Ollama endpoint below. Options:
- **Local testing**: `http://localhost:11434` (with kubectl port-forward)
- **KEDA HTTP Interceptor**: Get the interceptor IP and use with hostname resolution
- **Environment variable**: Set `OLLAMA_ENDPOINT` before running

In [ ]:
# =============================================================================
# STEP 2: API CONNECTIVITY TEST
# =============================================================================
# Test connection to Ollama API by fetching the list of available models.
# This also serves as a health check to ensure the service is running and accessible.

print("Step 2: Testing Connection to Ollama API")
print("=" * 50)
print("[INFO] Checking if Ollama is reachable...\n")

try:
    # Measure response time to understand API latency
    # First request may be slow if pod is scaled to zero (KEDA scale-up takes 2-4 min)
    print("[WAIT] Sending health check request (first request may take 2-4 minutes)...")
    start_time = time.time()
    
    # GET /api/tags returns list of all models and their metadata
    response = requests.get(
        f"{OLLAMA_ENDPOINT}/api/tags",
        timeout=TIMEOUT
    )
    
    elapsed = time.time() - start_time
    
    if response.status_code == 200:
        print(f"[SUCCESS] Ollama is reachable (response time: {elapsed:.2f}s)")
        models = response.json()
        print(f"\n[INFO] Available Models:")
        
        if models.get('models'):
            # Display each model with its size in GB for reference
            for model in models['models']:
                model_name = model.get('name', 'Unknown')
                model_size = model.get('size', 0)
                # Convert bytes to GB for readability
                print(f"  * {model_name} ({model_size / (1024**3):.2f} GB)")
        else:
            # No models loaded yet - guide user to seed the cluster
            print("  No models loaded yet. Run seed-initial-model.sh to pull models.")
    else:
        # Non-200 status code indicates API error
        print(f"[ERROR] Received status code {response.status_code}")
        print(f"Response: {response.text}")
        
# Exception handlers for common failure modes:
except requests.exceptions.Timeout:
    # Timeout usually means pod is scaling up or endpoint is unreachable
    print(f"[ERROR] Timeout: Ollama didn't respond within {TIMEOUT} seconds")
    print("   Possible causes:")
    print("   * Pod is still scaling up (wait 2-4 minutes for first request)")
    print("   * Endpoint is incorrect")
    print("   * Network connectivity issue")
    
except requests.exceptions.ConnectionError as e:
    # Connection refused or host unreachable
    print(f"[ERROR] Connection Failed: {e}")
    print(f"   Check if endpoint is correct: {OLLAMA_ENDPOINT}")
    
except Exception as e:
    # Catch-all for unexpected errors
    print(f"[ERROR] {type(e).__name__}: {e}")

## Step 2: Connection Test

Test HTTP connectivity to the Ollama API and list available models.

**First Request Behavior:** 
If the KEDA autoscaler has scaled the deployment to zero replicas (saves costs when idle), 
the first request will trigger a scale-up. This pod provisioning takes 2-4 minutes on Spot VMs. 
Subsequent requests after the pod is running are much faster (typically <1 second).

**Expected Output:**
- Response time in seconds
- List of available models with their sizes in GB
- If no models are listed, run `./scripts/seed-initial-model.sh` to pull the initial model

In [ ]:
# =============================================================================
# STEP 3: INFERENCE TEST
# =============================================================================
# Perform a test inference request to verify model loading and response generation.
# This demonstrates the complete request/response cycle with performance metrics.

print("Step 3: Inference Test")
print("=" * 50)

# Define the prompt to send to the model
# This should be a question or instruction the model will respond to
prompt = "Explain why using Kubernetes Spot VMs is a smart choice for a Linux administrator."

# Construct the inference request payload
# All parameters are optional - shown here are the most common ones
payload = {
    "model": MODEL_NAME,           # Which model to use
    "prompt": prompt,               # The input text/question
    "stream": False,                # False: wait for full response (True: stream tokens as they appear)
    "temperature": 0.7,             # 0.0=deterministic, 1.0=creative (controls response randomness)
    "num_predict": 256              # Maximum number of tokens to generate in response
}

print(f"[INFO] Sending inference request...")
print(f"   Model: {MODEL_NAME}")
print(f"   Prompt length: {len(prompt)} characters")
print(f"   Parameters: temperature={payload['temperature']}, max_tokens={payload['num_predict']}")
print()

try:
    # Record start time for performance measurement
    start_time = time.time()
    
    # POST request to /api/generate with the inference payload
    response = requests.post(
        f"{OLLAMA_ENDPOINT}/api/generate",
        json=payload,
        timeout=TIMEOUT
    )
    
    # Calculate total request duration
    end_time = time.time()
    elapsed = end_time - start_time
    
    if response.status_code == 200:
        # Successful inference - parse and display results
        result = response.json()
        
        print(f"[SUCCESS] Inference successful!")
        print(f"[METRICS]")
        print(f"   Total time: {elapsed:.2f}s")
        print(f"   Tokens generated: {result.get('eval_count', 'N/A')}")  # Actual tokens produced
        print(f"   Prompt tokens: {result.get('prompt_eval_count', 'N/A')}")  # Tokens in input
        print(f"\n[RESPONSE]\n")
        print("-" * 50)
        # Display the actual model response
        print(result['response'])
        print("-" * 50)
    else:
        # API error - display status and response for debugging
        print(f"[ERROR] Inference Failed")
        print(f"   Status Code: {response.status_code}")
        print(f"   Response: {response.text}")
        
# Exception handlers cover common failure scenarios:
except requests.exceptions.Timeout:
    # Request took too long - model may be overloaded or unresponsive
    print(f"[ERROR] Timeout: Inference took longer than {TIMEOUT} seconds")
    
except requests.exceptions.ConnectionError as e:
    # Network error - endpoint unreachable
    print(f"[ERROR] Connection Failed: {e}")
    
except Exception as e:
    # Unexpected error - display type and details for debugging
    print(f"[ERROR] {type(e).__name__}: {e}")

In [ ]:
# =============================================================================
# ADVANCED INFERENCE EXAMPLES
# =============================================================================
# Demonstrate different use cases with varied parameters:
# - Code generation: Use low temperature for deterministic, syntactically correct output
# - Creative writing: Use high temperature for diverse, creative responses

print("Advanced Inference Examples")
print("=" * 50)

# =============================================================================
# Example 1: CODE GENERATION
# =============================================================================
# Lower temperature (0.3) produces more deterministic, consistent code
# Higher num_predict allows for longer code blocks

print("\nExample 1: Code Generation")
print("-" * 50)

code_prompt = "Write a Python function to calculate the Fibonacci sequence up to n numbers."

code_payload = {
    "model": MODEL_NAME,
    "prompt": code_prompt,
    "stream": False,
    "temperature": 0.3,  # Low temperature: more deterministic, fewer hallucinations
    "num_predict": 512   # Increased limit for longer code output
}

try:
    response = requests.post(
        f"{OLLAMA_ENDPOINT}/api/generate",
        json=code_payload,
        timeout=TIMEOUT
    )
    
    if response.status_code == 200:
        result = response.json()
        # Display generated code
        print(result['response'])
    else:
        print(f"Error: {response.status_code}")
        
except Exception as e:
    print(f"Error: {e}")

# =============================================================================
# Example 2: CREATIVE WRITING
# =============================================================================
# Higher temperature (0.9) produces more diverse and creative responses
# Each run will generate different output due to the increased randomness

print("\n" + "=" * 50)
print("Example 2: Creative Writing")
print("-" * 50)

creative_prompt = "Write a short poem (4 lines) about cloud computing."

creative_payload = {
    "model": MODEL_NAME,
    "prompt": creative_prompt,
    "stream": False,
    "temperature": 0.9,  # High temperature: more creative, more variation between runs
    "num_predict": 256   # Sufficient for short poem
}

try:
    response = requests.post(
        f"{OLLAMA_ENDPOINT}/api/generate",
        json=creative_payload,
        timeout=TIMEOUT
    )
    
    if response.status_code == 200:
        result = response.json()
        # Display generated poem
        print(result['response'])
    else:
        print(f"Error: {response.status_code}")
        
except Exception as e:
    print(f"Error: {e}")

## Step 4: Advanced Inference Examples

Test different use cases with optimized parameters for each scenario.

**Key Differences Between Examples:**

1. **Code Generation (temperature=0.3)**
   - Low temperature ensures consistent, syntactically correct code
   - Fewer hallucinations and creative deviations
   - Best for: Code snippets, SQL queries, technical content

2. **Creative Writing (temperature=0.9)**
   - High temperature produces diverse, creative output
   - Each run generates different content
   - Best for: Poetry, stories, brainstorming, creative content

**Try Your Own:**
Modify the `prompt` and `temperature` values to experiment with different use cases!

## Step 3: Inference Test

Run a complete inference request to generate text from the Ollama model.

**What This Tests:**
- Model responsiveness and inference capability
- Response time and token generation metrics
- Full API integration (request → processing → response)

**Parameters Explained:**
- **temperature**: Controls randomness (0.0=consistent, 1.0=creative). Use 0.0-0.3 for code/facts, 0.7-1.0 for creative content
- **num_predict**: Maximum tokens to generate. Increase for longer responses, decrease to limit output
- **stream**: Set to True to see tokens appear in real-time; False waits for complete response

**Metrics:**
- **Total time**: End-to-end request duration (includes network latency and inference time)
- **Tokens generated**: Number of response tokens produced
- **Prompt tokens**: Number of tokens in your input prompt